# Position Bias Exploration Benchmark

This notebook extends the week 1 position-bias exercise into a more deliberate benchmark.

What it does:
- focuses on `openai/gpt-5-codex-mini` served through Quotio
- uses harder integer arithmetic instead of only simple add/subtract
- compares 4-choice and 6-choice multiple-choice settings
- checks randomized vs fixed correct-answer positions across all slots
- runs 28 questions per condition by default
- computes a short statistical check-up and produces charts

Notes:
- this notebook writes its logs to `w1/position_bias_exploration_logs/`
- figures are saved to `w1/position_bias_outputs/`
- the focus is position bias, so the solver uses standard `multiple_choice()` rather than CoT to avoid scorer-format noise

## Experiment design

For the model and each option count:
1. generate one shared set of harder arithmetic questions
2. run one randomized-position condition
3. run one fixed-position condition for every answer slot
4. summarize accuracy by condition
5. inspect whether randomized chosen letters skew away from uniform
6. visualize fixed-position sensitivity and randomized answer distributions

Default settings are intentionally conservative enough to run on a laptop, but large enough to be more informative than the original toy benchmark.

In [2]:
import json
import math
import os
import random
import re
import string
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display

from inspect_ai import Task, eval, task
from inspect_ai.dataset import Sample
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, system_message


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "w1").exists():
            return candidate
    if cwd.name == "w1" and (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    return cwd


PROJECT_ROOT = resolve_project_root()
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

QUOTIO_BASE_URL = os.environ.get("QUOTIO_BASE_URL", "http://127.0.0.1:8317/v1")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ["QUOTIO_API_KEY"]
os.environ.setdefault("OPENAI_BASE_URL", QUOTIO_BASE_URL)
os.environ.setdefault("QUOTIO_BASE_URL", QUOTIO_BASE_URL)

BASE_URL = os.environ["QUOTIO_BASE_URL"]
LOG_DIR = PROJECT_ROOT / "w1" / "position_bias_exploration_logs"
OUTPUT_DIR = PROJECT_ROOT / "w1" / "position_bias_outputs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    {
        "label": "gpt-5-codex-mini",
        "model": "openai/gpt-5-codex-mini",
        "base_url": BASE_URL,
    },
]

OPTION_COUNTS = [4, 6]
N_QUESTIONS = 28
BASE_SEED = 42
SIMULATIONS = 5_000

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print(f"Project root: {PROJECT_ROOT}")
print(f"Quotio base URL: {BASE_URL}")
print(f"Logging to: {LOG_DIR}")
print(f"Figures to: {OUTPUT_DIR}")
pd.DataFrame(MODEL_SPECS)

Quotio base URL: http://127.0.0.1:8317/v1
Logging to: w1/position_bias_exploration_logs
Figures to: w1/position_bias_outputs


,label,model,base_url
0,gpt-5-codex-mini,openai/gpt-5-codex-mini,http://127.0.0.1:8317/v1


In [ ]:
def answer_letters(option_count: int) -> list[str]:
    return list(string.ascii_uppercase[:option_count])


def fixed_position_label(position: int | None, option_count: int) -> str:
    if position is None:
        return "randomized"
    return f"fixed_{answer_letters(option_count)[position]}"


def make_rng(seed: int) -> random.Random:
    return random.Random(seed)


def generate_harder_questions(n: int, seed: int = BASE_SEED) -> list[dict]:
    """Generate harder integer arithmetic questions with exact integer answers."""
    rng = make_rng(seed)
    records: list[dict] = []
    kinds = [
        "add_sub_chain",
        "mul_add",
        "mul_sub",
        "exact_division",
        "percent_plus",
        "order_of_ops",
    ]

    while len(records) < n:
        kind = rng.choice(kinds)

        if kind == "add_sub_chain":
            a = rng.randint(50_000, 900_000)
            b = rng.randint(10_000, 200_000)
            c = rng.randint(10_000, 200_000)
            answer = a - b + c
            prompt = f"Compute exactly: ({a} - {b}) + {c}"

        elif kind == "mul_add":
            a = rng.randint(120, 999)
            b = rng.randint(12, 99)
            c = rng.randint(1_000, 25_000)
            answer = a * b + c
            prompt = f"Compute exactly: ({a} × {b}) + {c}"

        elif kind == "mul_sub":
            a = rng.randint(100, 999)
            b = rng.randint(11, 87)
            c = rng.randint(500, 8_000)
            answer = a * b - c
            prompt = f"Compute exactly: ({a} × {b}) - {c}"

        elif kind == "exact_division":
            divisor = rng.randint(7, 49)
            quotient = rng.randint(200, 4_000)
            extra = rng.randint(30, 900)
            dividend = divisor * quotient
            answer = quotient + extra
            prompt = f"Compute exactly: ({dividend} ÷ {divisor}) + {extra}"

        elif kind == "percent_plus":
            percent = rng.choice([5, 10, 12, 15, 20, 25, 30, 40, 50, 75])
            factor = rng.randint(40, 600)
            base = factor * 100
            extra = rng.randint(75, 5_000)
            answer = (base * percent) // 100 + extra
            prompt = f"Compute exactly: {percent}% of {base}, then add {extra}"

        elif kind == "order_of_ops":
            a = rng.randint(25, 250)
            b = rng.randint(11, 99)
            c = rng.randint(8, 45)
            d = rng.randint(5, 30)
            answer = (a + b) * c - d
            prompt = f"Compute exactly: ({a} + {b}) × {c} - {d}"

        else:
            raise ValueError(f"Unknown kind: {kind}")

        records.append(
            {
                "id": len(records) + 1,
                "kind": kind,
                "input": prompt,
                "answer": int(answer),
            }
        )

    return records


questions = generate_harder_questions(N_QUESTIONS)
pd.DataFrame(questions).head(8)

In [ ]:
def generate_distractors(correct: int, n: int, seed: int) -> list[str]:
    """Generate plausible numeric distractors near the correct answer."""
    rng = make_rng(seed)
    magnitude = max(3, abs(correct))
    step_small = max(1, magnitude // 100)
    step_medium = max(2, magnitude // 25)
    step_large = max(5, magnitude // 10)

    candidate_values = {
        correct + 1,
        correct - 1,
        correct + step_small,
        correct - step_small,
        correct + step_medium,
        correct - step_medium,
        correct + step_large,
        correct - step_large,
        int(round(correct * 1.05)),
        int(round(correct * 0.95)),
        int(round(correct * 1.10)),
        int(round(correct * 0.90)),
    }

    if correct != 0:
        candidate_values.add(-correct)

    candidate_values = [value for value in candidate_values if value != correct]
    rng.shuffle(candidate_values)

    distractors: list[str] = []
    used = {str(correct)}
    for value in candidate_values:
        as_str = str(value)
        if as_str not in used:
            distractors.append(as_str)
            used.add(as_str)
        if len(distractors) == n:
            return distractors

    while len(distractors) < n:
        jitter = rng.randint(-step_large * 2, step_large * 2)
        proposal = str(correct + jitter)
        if proposal not in used:
            distractors.append(proposal)
            used.add(proposal)

    return distractors


def create_samples(
    questions: list[dict],
    option_count: int,
    correct_position: int | None,
    seed: int = BASE_SEED,
) -> list[Sample]:
    labels = answer_letters(option_count)
    samples: list[Sample] = []

    for idx, record in enumerate(questions):
        correct = str(record["answer"])
        distractors = generate_distractors(
            record["answer"],
            n=option_count - 1,
            seed=seed + option_count * 10_000 + idx,
        )
        options = distractors.copy()

        if correct_position is None:
            options.append(correct)
            shuffle_rng = make_rng(seed + option_count * 20_000 + idx)
            shuffle_rng.shuffle(options)
            target_letter = labels[options.index(correct)]
        else:
            options.insert(correct_position, correct)
            target_letter = labels[correct_position]

        samples.append(
            Sample(
                id=record["id"],
                input=record["input"],
                choices=options,
                target=target_letter,
                metadata={
                    "kind": record["kind"],
                    "numeric_answer": record["answer"],
                    "option_count": option_count,
                    "correct_position": correct_position,
                },
            )
        )

    return samples


@task
def position_bias_exploration(
    questions: list[dict],
    option_count: int,
    correct_position: int | None = None,
):
    return Task(
        dataset=create_samples(questions, option_count=option_count, correct_position=correct_position),
        solver=[
            system_message(
                "Solve the arithmetic exactly and choose the single best answer option."
            ),
            multiple_choice(),
        ],
        scorer=choice(),
    )


sample_preview = create_samples(questions[:3], option_count=6, correct_position=5)
pd.DataFrame(
    [
        {
            "input": sample.input,
            "choices": sample.choices,
            "target": sample.target,
        }
        for sample in sample_preview
    ]
)

In [ ]:
def experiment_conditions(option_count: int) -> list[int | None]:
    return [None] + list(range(option_count))


def run_experiment_matrix(
    questions: list[dict],
    models: list[dict],
    option_counts: list[int],
    n_questions: int = N_QUESTIONS,
) -> list:
    logs = []

    for model_spec in models:
        for option_count in option_counts:
            for correct_position in experiment_conditions(option_count):
                condition_label = fixed_position_label(correct_position, option_count)
                print(
                    f"Running model={model_spec['label']}, option_count={option_count}, condition={condition_label}, questions={n_questions}"
                )
                run_logs = eval(
                    position_bias_exploration,
                    model=model_spec["model"],
                    model_base_url=model_spec["base_url"],
                    task_args={
                        "questions": questions[:n_questions],
                        "option_count": option_count,
                        "correct_position": correct_position,
                    },
                    metadata={
                        "model_label": model_spec["label"],
                        "model_name": model_spec["model"],
                        "option_count": option_count,
                        "correct_position": correct_position,
                        "condition_label": condition_label,
                        "n_questions": n_questions,
                        "benchmark": "position_bias_exploration",
                    },
                    log_dir=str(LOG_DIR),
                    display="none",
                )
                logs.extend(run_logs)

    return logs

In [ ]:
# This cell runs the full experiment matrix.
# Defaults:
# - 1 model
# - option counts: 4 and 6
# - conditions per option count: randomized + every fixed slot
# - 28 questions per condition

# If

experiment_logs = run_experiment_matrix(
    questions=questions,
    models=MODEL_SPECS,
    option_counts=OPTION_COUNTS,
    n_questions=N_QUESTIONS,
)

print(f"Completed {len(experiment_logs)} eval run(s).")

In [3]:
def coerce_log_path(location) -> Path:
    text = str(location)
    if text.startswith("file://"):
        return Path(urlparse(text).path)
    return Path(text)


def extract_predicted_letter(score_payload: dict, option_count: int) -> str | None:
    labels = set(answer_letters(option_count))
    answer = score_payload.get("answer")
    if isinstance(answer, str):
        cleaned = answer.strip().replace("$", "")
        if cleaned in labels:
            return cleaned

    explanation = score_payload.get("explanation", "") or ""
    match = re.search(r"ANSWER:\s*\$?([A-Z])", explanation)
    if match and match.group(1) in labels:
        return match.group(1)

    return None


def read_eval_run(log_path: Path) -> tuple[dict, list[dict]]:
    with zipfile.ZipFile(log_path) as zf:
        header = json.loads(zf.read("header.json"))
        summaries = json.loads(zf.read("summaries.json"))
    return header, summaries


def build_sample_frame(logs: list) -> pd.DataFrame:
    rows = []

    for log in logs:
        log_path = coerce_log_path(log.location)
        header, summaries = read_eval_run(log_path)
        metadata = header.get("eval", {}).get("metadata", {})
        option_count = int(metadata["option_count"])

        for row in summaries:
            score_name, score_payload = next(iter(row["scores"].items()))
            rows.append(
                {
                    "log_path": str(log_path),
                    "model_label": metadata["model_label"],
                    "model_name": metadata["model_name"],
                    "option_count": option_count,
                    "correct_position": metadata["correct_position"],
                    "condition_label": metadata["condition_label"],
                    "sample_id": row["id"],
                    "input": row["input"],
                    "target": row["target"],
                    "predicted": extract_predicted_letter(score_payload, option_count),
                    "score_name": score_name,
                    "score_value": score_payload.get("value"),
                    "correct": score_payload.get("value") == "C",
                    "kind": row.get("metadata", {}).get("kind"),
                    "working_time": row.get("working_time"),
                    "input_tokens": next(iter(row.get("model_usage", {}).values()), {}).get("input_tokens"),
                    "output_tokens": next(iter(row.get("model_usage", {}).values()), {}).get("output_tokens"),
                    "total_tokens": next(iter(row.get("model_usage", {}).values()), {}).get("total_tokens"),
                }
            )

    sample_df = pd.DataFrame(rows)
    sample_df["fixed_position_label"] = sample_df.apply(
        lambda r: fixed_position_label(r["correct_position"], int(r["option_count"])),
        axis=1,
    )
    return sample_df


sample_df = build_sample_frame(experiment_logs)
print(sample_df.shape)
sample_df.head()

NameError: name 'experiment_logs' is not defined

In [ ]:
def wilson_interval(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    margin = z * math.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return (center - margin, center + margin)


def monte_carlo_uniform_pvalue(counts: list[int], simulations: int = SIMULATIONS, seed: int = BASE_SEED) -> tuple[float, float]:
    observed = np.array(counts, dtype=float)
    n = int(observed.sum())
    k = len(observed)
    if n == 0 or k <= 1:
        return (float("nan"), float("nan"))

    expected = n / k
    chi2 = float(((observed - expected) ** 2 / expected).sum())
    rng = np.random.default_rng(seed)
    sims = rng.multinomial(n, [1 / k] * k, size=simulations)
    sim_stats = ((sims - expected) ** 2 / expected).sum(axis=1)
    p_value = float((np.count_nonzero(sim_stats >= chi2) + 1) / (simulations + 1))
    return chi2, p_value


def summarize_runs(sample_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    group_cols = ["model_label", "option_count", "condition_label", "correct_position", "fixed_position_label"]
    for keys, group in sample_df.groupby(group_cols, dropna=False):
        model_label, option_count, condition_label, correct_position, fixed_position_label = keys
        correct_n = int(group["correct"].sum())
        n = len(group)
        ci_low, ci_high = wilson_interval(correct_n, n)
        records.append(
            {
                "model_label": model_label,
                "option_count": option_count,
                "condition_label": condition_label,
                "correct_position": correct_position,
                "fixed_position_label": fixed_position_label,
                "n": n,
                "correct_n": correct_n,
                "accuracy": correct_n / n,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "avg_working_time": group["working_time"].mean(),
                "avg_total_tokens": group["total_tokens"].mean(),
            }
        )
    return pd.DataFrame(records).sort_values(["model_label", "option_count", "condition_label"])


def randomized_position_stats(sample_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    randomized = sample_df[sample_df["condition_label"] == "randomized"].copy()
    for keys, group in randomized.groupby(["model_label", "option_count"]):
        model_label, option_count = keys
        labels = answer_letters(int(option_count))
        counts = [int((group["predicted"] == label).sum()) for label in labels]
        chi2, p_value = monte_carlo_uniform_pvalue(counts)
        n = len(group)
        cramers_v = math.sqrt(chi2 / (n * (option_count - 1))) if n and option_count > 1 else float("nan")
        row = {
            "model_label": model_label,
            "option_count": option_count,
            "n": n,
            "chi2_uniform": chi2,
            "p_value_uniform": p_value,
            "cramers_v": cramers_v,
        }
        row.update({f"count_{label}": count for label, count in zip(labels, counts)})
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["model_label", "option_count"])


def fixed_position_gaps(run_summary: pd.DataFrame) -> pd.DataFrame:
    fixed = run_summary[run_summary["condition_label"] != "randomized"].copy()
    rows = []
    for keys, group in fixed.groupby(["model_label", "option_count"]):
        model_label, option_count = keys
        rows.append(
            {
                "model_label": model_label,
                "option_count": option_count,
                "min_accuracy": group["accuracy"].min(),
                "max_accuracy": group["accuracy"].max(),
                "accuracy_gap": group["accuracy"].max() - group["accuracy"].min(),
            }
        )
    return pd.DataFrame(rows).sort_values(["model_label", "option_count"])


run_summary = summarize_runs(sample_df)
randomized_stats = randomized_position_stats(sample_df)
fixed_gaps = fixed_position_gaps(run_summary)

print("Run summary")
display(run_summary)
print("\nRandomized position stats")
display(randomized_stats)
print("\nFixed-position accuracy gaps")
display(fixed_gaps)

In [ ]:
def print_brief_findings(run_summary: pd.DataFrame, randomized_stats: pd.DataFrame, fixed_gaps: pd.DataFrame) -> None:
    print("Short check-up")
    print("--------------")

    for _, row in fixed_gaps.iterrows():
        print(
            f"{row['model_label']} with {int(row['option_count'])} options: fixed-position accuracy ranged from {row['min_accuracy']:.3f} to {row['max_accuracy']:.3f} (gap={row['accuracy_gap']:.3f})"
        )

    print()
    for _, row in randomized_stats.iterrows():
        significance = "likely skew" if row["p_value_uniform"] < 0.05 else "no strong skew"
        print(
            f"{row['model_label']} with {int(row['option_count'])} options: chi2={row['chi2_uniform']:.2f}, Monte Carlo p≈{row['p_value_uniform']:.4f}, Cramér's V={row['cramers_v']:.3f} -> {significance}"
        )


print_brief_findings(run_summary, randomized_stats, fixed_gaps)

In [ ]:
plot_df = run_summary.copy()
plot_df["condition_plot"] = plot_df["fixed_position_label"]

fig, axes = plt.subplots(len(OPTION_COUNTS), 1, figsize=(16, 5 * len(OPTION_COUNTS)), constrained_layout=True)
if len(OPTION_COUNTS) == 1:
    axes = [axes]

for ax, option_count in zip(axes, OPTION_COUNTS):
    subset = plot_df[plot_df["option_count"] == option_count]
    sns.barplot(data=subset, x="condition_plot", y="accuracy", hue="model_label", ax=ax)
    ax.set_title(f"Accuracy by condition ({option_count} options)")
    ax.set_xlabel("Condition")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=30)

accuracy_path = OUTPUT_DIR / "accuracy_by_condition.png"
fig.savefig(accuracy_path, bbox_inches="tight")
plt.show()
print(f"Saved {accuracy_path}")

In [ ]:
fixed_df = run_summary[run_summary["condition_label"] != "randomized"].copy()

for option_count in OPTION_COUNTS:
    subset = fixed_df[fixed_df["option_count"] == option_count].copy()
    if subset.empty:
        continue

    pivot = subset.pivot(index="model_label", columns="fixed_position_label", values="accuracy")
    ordered_columns = [fixed_position_label(i, option_count) for i in range(option_count)]
    pivot = pivot.reindex(columns=ordered_columns)

    fig, ax = plt.subplots(figsize=(2 + 2 * option_count, 3.5))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"Fixed-position accuracy heatmap ({option_count} options)")
    ax.set_xlabel("Correct answer slot")
    ax.set_ylabel("Model")
    path = OUTPUT_DIR / f"fixed_position_heatmap_{option_count}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved {path}")

In [ ]:
randomized = sample_df[sample_df["condition_label"] == "randomized"].copy()
choice_share = (
    randomized.groupby(["model_label", "option_count", "predicted"]).size().rename("count").reset_index()
)
choice_share["share"] = choice_share.groupby(["model_label", "option_count"])["count"].transform(lambda s: s / s.sum())

fig, axes = plt.subplots(len(OPTION_COUNTS), 1, figsize=(16, 4.5 * len(OPTION_COUNTS)), constrained_layout=True)
if len(OPTION_COUNTS) == 1:
    axes = [axes]

for ax, option_count in zip(axes, OPTION_COUNTS):
    subset = choice_share[choice_share["option_count"] == option_count]
    sns.barplot(data=subset, x="predicted", y="share", hue="model_label", ax=ax)
    ax.axhline(1 / option_count, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Randomized chosen-letter shares ({option_count} options)")
    ax.set_xlabel("Predicted letter")
    ax.set_ylabel("Share of predictions")
    ax.set_ylim(0, max(0.35, 1 / option_count + 0.15))

choice_path = OUTPUT_DIR / "randomized_choice_shares.png"
fig.savefig(choice_path, bbox_inches="tight")
plt.show()
print(f"Saved {choice_path}")

## How to interpret the outputs

Use the three outputs together:
- **accuracy by condition** tells you whether some fixed answer slots are systematically easier or harder for a model
- **fixed-position heatmaps** make slot sensitivity obvious at a glance
- **randomized chosen-letter shares + chi-square check** tell you whether the model's answer choices skew toward particular letters when the correct slot is randomized

A model with strong position bias would usually show one or both of:
1. materially different accuracy by fixed correct-answer slot
2. randomized chosen-letter shares that skew away from uniform beyond what you'd expect from noise

This notebook is still a small benchmark, so treat it as an exploratory signal rather than a publication-grade estimate.